# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessing metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore record sets and their fields

print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"- RecordSet Name: {record_set.name}, @id: {record_set.id}")
    print("  Fields and Columns:")
    for field in record_set.fields:
        print(f"    - Field: {field.name}, @id: {field.id}, dataType: {field.data_type}, column @id: {field.column.id if hasattr(field, 'column') and field.column else 'N/A'}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all RecordSet @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print("RecordSet @ids for extraction:")
print(record_set_ids)

# Load all record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found in RecordSet {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demo purposes, select a numeric field such as GAD-7 score (commonly available in mental health survey datasets).
# Please adjust 'gad7_score' and 'demographics' below to the correct @id or field name in your dataset as per the printed overview above.

# Example RecordSet and Field IDs (replace with actual IDs from overview)
example_record_set_id = record_set_ids[0] if record_set_ids else None
example_numeric_field = None
group_field = None

if example_record_set_id and example_record_set_id in dataframes:
    cols = dataframes[example_record_set_id].columns
    # Heuristically pick a numeric field like 'gad7_score', 'phq9_score', 'psq_score', or fallback to the first numeric column
    for col in cols:
        if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower():
            example_numeric_field = col
            break
    # Fallback to first numeric column
    if example_numeric_field is None:
        numeric_cols = dataframes[example_record_set_id].select_dtypes(include='number').columns
        if len(numeric_cols) > 0:
            example_numeric_field = numeric_cols[0]
    # Try a reasonable group field
    for col in cols:
        if col.lower() in ['village', 'gender', 'sex', 'level_of_education', 'marital_status']:
            group_field = col
            break
    print(f"Numeric field for EDA: {example_numeric_field}")
    print(f"Group field for EDA: {group_field}")

    if example_numeric_field:
        # Clean out NaNs (if any)
        df = dataframes[example_record_set_id].copy()
        df = df[pd.notnull(df[example_numeric_field])]

        threshold = df[example_numeric_field].mean()  # Use mean as demo threshold
        filtered_df = df[df[example_numeric_field] > threshold]
        print(f"Filtered records with {example_numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize field
        filtered_df[f"{example_numeric_field}_normalized"] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
        print(f"Normalized {example_numeric_field} for filtered records:")
        print(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[example_numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {example_numeric_field}):")
            print(grouped_df)
    else:
        print("No numeric field detected for EDA. Please check your dataset.")
else:
    print("No populated record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if the analysis data is present
if example_record_set_id and example_record_set_id in dataframes and example_numeric_field:
    df = dataframes[example_record_set_id].copy()
    df = df[pd.notnull(df[example_numeric_field])]

    plt.figure(figsize=(8, 4))
    sns.histplot(df[example_numeric_field], kde=True, bins=20)
    plt.title(f"Distribution of {example_numeric_field}")
    plt.xlabel(example_numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=example_numeric_field, data=df)
        plt.title(f"{example_numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded the Kilifi County FAIR² Mental Health Survey dataset, previewed its record sets and fields, and performed basic EDA including data filtering, normalization, grouping, and simple visualizations. The approach demonstrated how to reference dataset entities using their `@id` fields via the mlcroissant interface. For deeper insights, you may augment the analysis using further domain-specific fields or combine with external health or demographic datasets.*